In [ ]:
import os
import csv
from pathlib import Path

def is_header_row(row: list, reference_header: list) -> bool:
    """Détecte si une ligne est un en-tête (identique ou très similaire à l'en-tête de référence)"""
    if not row or not reference_header:
        return False
    
    # Normaliser pour comparaison (majuscules, sans espaces)
    normalized_row = [str(cell).strip().upper() for cell in row]
    normalized_header = [str(cell).strip().upper() for cell in reference_header]
    
    # Si exactement identique
    if normalized_row == normalized_header:
        return True
    
    # Si au moins 80% des colonnes correspondent
    if len(normalized_row) == len(normalized_header):
        matches = sum(1 for a, b in zip(normalized_row, normalized_header) if a == b)
        similarity = matches / len(normalized_header)
        if similarity >= 0.8:
            return True
    
    # Vérifier si contient des mots-clés typiques d'en-tête
    header_keywords = ['TYPE', 'CODE', 'ISIN', 'DESIGNATION', 'COURS', 'EMETTEUR', 
                       'MONTANT', 'DATE', 'STATUT', 'INSTRUMENTS', 'QUANTITE', 
                       'VOLUME', 'INDICES', 'VALEUR', 'VARIATION']
    
    row_text = ' '.join(normalized_row).upper()
    matching_keywords = sum(1 for kw in header_keywords if kw in row_text)
    
    # Si au moins 4 mots-clés d'en-tête dans la ligne, c'est probablement un en-tête
    if matching_keywords >= 4:
        return True
    
    return False

def merge_csv_files(category_path: str, category_name: str, output_file: str) -> bool:
    """Fusionne tous les fichiers CSV d'une catégorie EN SUPPRIMANT TOUS LES EN-TÊTES DUPLIQUÉS"""
    print(f"\n Fusion: {category_name}")
    print("-" * 70)
    
    # Récupérer tous les fichiers CSV
    csv_files = sorted(Path(category_path).glob("*.csv"))
    
    if not csv_files:
        print(f"     Aucun fichier CSV trouvé")
        return False
    
    print(f"    {len(csv_files)} fichier(s) à fusionner")
    
    merged_header = None
    all_rows = []
    headers_skipped = 0
    
    # Lire tous les fichiers CSV
    for idx, csv_file in enumerate(csv_files, 1):
        try:
            with open(csv_file, 'r', encoding='utf-8') as f:
                reader = csv.reader(f)
                rows = list(reader)
                
                if not rows:
                    continue
                
                # Premier fichier : définir l'en-tête de référence
                if merged_header is None:
                    merged_header = rows[0]
                    print(f"    En-tête principal: {len(merged_header)} colonne(s)")
                    print(f"    Colonnes: {', '.join(merged_header[:5])}{'...' if len(merged_header) > 5 else ''}")
                
                # Traiter toutes les lignes SAUF la première (qui est l'en-tête)
                for row_idx, row in enumerate(rows[1:], start=2):  # start=2 car ligne 1 = en-tête
                    # Vérifier si cette ligne est un en-tête dupliqué
                    if is_header_row(row, merged_header):
                        headers_skipped += 1
                        print(f"        En-tête dupliqué IGNORÉ (fichier {csv_file.name}, ligne {row_idx})")
                        continue
                    
                    # Vérifier que ce n'est pas une ligne vide
                    if not any(cell.strip() for cell in row):
                        continue
                    
                    # C'est une ligne de données valide
                    all_rows.append(row)
                
                data_count = len([r for r in rows[1:] if any(cell.strip() for cell in r)])
                print(f"   [{idx}/{len(csv_files)}] {csv_file.name}: {data_count} ligne(s) de données")
        
        except Exception as e:
            print(f"     Erreur lecture {csv_file.name}: {e}")
            continue
    
    if merged_header is None or not all_rows:
        print(f"    Aucune donnée à fusionner")
        return False
    
    print(f"\n    Total en-têtes dupliqués supprimés: {headers_skipped}")
    
    # Écrire le fichier fusionné
    try:
        with open(output_file, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
            
            # Écrire l'en-tête UNE SEULE FOIS
            writer.writerow(merged_header)
            
            # Écrire toutes les données (déjà filtrées, sans en-têtes)
            for row in all_rows:
                # Normaliser le nombre de colonnes
                if len(row) < len(merged_header):
                    row.extend([''] * (len(merged_header) - len(row)))
                elif len(row) > len(merged_header):
                    row = row[:len(merged_header)]
                
                writer.writerow(row)
        
        print(f"    Fusionné: {len(all_rows)} lignes de données totales")
        print(f"    Fichier: {os.path.basename(output_file)}")
        return True
        
    except Exception as e:
        print(f"    Erreur écriture: {e}")
        return False

def merge_all_categories(input_folder: str, output_folder: str):
    """Fusionne tous les CSV par catégorie"""
    
    # Vérifier que le dossier d'entrée existe
    if not os.path.exists(input_folder):
        print(f" Dossier introuvable: {input_folder}")
        return
    
    # Créer le dossier de sortie
    os.makedirs(output_folder, exist_ok=True)
    
    print(" FUSION DES CSV PAR CATÉGORIE (SANS EN-TÊTES DUPLIQUÉS)")
    print(f" Source: {os.path.abspath(input_folder)}")
    print(f" Sortie: {os.path.abspath(output_folder)}")
    print("=" * 70)
    
    categories_processed = 0
    categories_success = 0
    
    # Parcourir toutes les catégories
    for category in sorted(os.listdir(input_folder)):
        category_path = os.path.join(input_folder, category)
        
        if not os.path.isdir(category_path):
            continue
        
        categories_processed += 1
        
        # Nom du fichier de sortie
        output_file = os.path.join(output_folder, f"{category}_MERGED.csv")
        
        # Fusionner
        if merge_csv_files(category_path, category, output_file):
            categories_success += 1
    
    # Rapport final
    print("\n" + "=" * 70)
    print(" RAPPORT FINAL")
    print("=" * 70)
    print(f" Catégories fusionnées: {categories_success}/{categories_processed}")
    print(f"\n CSV fusionnés dans: {os.path.abspath(output_folder)}")
    
    # Lister les fichiers créés
    if os.path.exists(output_folder):
        print("\n Fichiers créés:")
        merged_files = sorted(Path(output_folder).glob("*.csv"))
        
        for mf in merged_files:
            size_kb = mf.stat().st_size / 1024
            
            # Compter les lignes
            with open(mf, 'r', encoding='utf-8') as f:
                line_count = sum(1 for _ in f) - 1  # -1 pour l'en-tête
            
            print(f"   • {mf.name:45s} {size_kb:8.1f} KB   {line_count:7,d} lignes")
    
    print("\n Fusion terminée!")

if __name__ == "__main__":
    #  CONFIGUREZ VOS CHEMINS ICI
    INPUT_PATH = input(" Entrez le chemin du dossier contenant les CSV (ex: D:/data/csv): ").strip()
    OUTPUT_PATH = input(" Entrez le chemin du dossier de sortie (ex: D:/data/merged): ").strip()
    
    # Nettoyer les guillemets si l'utilisateur a copié-collé un chemin Windows
    INPUT_PATH = INPUT_PATH.strip('"').strip("'")
    OUTPUT_PATH = OUTPUT_PATH.strip('"').strip("'")
    
    print("\n" + "=" * 70)
    merge_all_categories(INPUT_PATH, OUTPUT_PATH)


 FUSION DES CSV PAR CATÉGORIE (SANS EN-TÊTES DUPLIQUÉS)
 Source: D:\rouge_file\Analyse_du_performance_de_la_bourse_casablanca\DATA_BULLETINS_CSV_INDIVIDUAL
 Sortie: D:\rouge_file\Analyse_du_performance_de_la_bourse_casablanca\Merged

 Fusion: Indices
----------------------------------------------------------------------
    679 fichier(s) à fusionner
    En-tête principal: 5 colonne(s)
    Colonnes: Type, Indices, Valeur Indice, Variation / veille, Variation / 31/12
   [1/679] BCFR_20230403.csv: 31 ligne(s) de données
   [2/679] BCFR_20230407.csv: 32 ligne(s) de données
   [3/679] BCFR_20230410.csv: 32 ligne(s) de données
   [4/679] BCFR_20230411.csv: 32 ligne(s) de données
   [5/679] BCFR_20230412.csv: 32 ligne(s) de données
   [6/679] BCFR_20230413.csv: 32 ligne(s) de données
   [7/679] BCFR_20230414.csv: 32 ligne(s) de données
   [8/679] BCFR_20230417.csv: 32 ligne(s) de données
   [9/679] BCFR_20230418.csv: 32 ligne(s) de données
   [10/679] BCFR_20230419.csv: 32 ligne(s) de donné